# 03 · Put the tables in DuckDB and build the monthly view in SQL
Each table is loaded into `data`, written to DuckDB, then joined in SQL.

In [1]:
from pathlib import Path
import pandas as pd
import duckdb

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT = ROOT / "data"
con = duckdb.connect(str(OUT / "sip_pulse.duckdb"))   # small local SQL database (not committed; rebuilt each run)

### Load equity flows

In [2]:
data = pd.read_parquet(OUT / "equity.parquet").reset_index()           # one row per month
con.execute("CREATE OR REPLACE TABLE equity AS SELECT * FROM data")
data.head(3)

,month,equity_inflow_cr,equity_aum_cr
0,2019-04,4608.741045,703549.778721
1,2019-05,5407.750458,724033.958682
2,2019-06,7663.138036,723729.274165


### Load SIP contributions

In [3]:
data = pd.read_parquet(OUT / "sip.parquet").reset_index()
con.execute("CREATE OR REPLACE TABLE sip AS SELECT * FROM data")
data.tail(3)

,month,sip_cr
124,2026-08,32297.0
125,2026-09,32490.0
126,2026-10,32160.0


### Load Nifty 50 month-end closes

In [4]:
data = pd.read_csv(OUT / "raw" / "nifty" / "nifty50_monthly.csv", dtype={"month": str})
con.execute("CREATE OR REPLACE TABLE nifty AS SELECT month, nifty_close FROM data")
data.tail(3)

,month,nifty_close,last_day
89,2026-08,24080.400391,2026-08-31
90,2026-09,22620.449219,2026-09-30
91,2026-10,23053.099609,2026-10-30


### Load groups and equity categories

In [5]:
data = pd.read_parquet(OUT / "groups.parquet")
con.execute("CREATE OR REPLACE TABLE groups AS SELECT * FROM data")
data = pd.read_parquet(OUT / "equity_categories.parquet")
con.execute("CREATE OR REPLACE TABLE equity_categories AS SELECT * FROM data")
print("loaded")

loaded


### The monthly view
One row per month with equity inflow, SIP contribution and the Nifty, plus:
- `nifty_return_pct`: Nifty change from the previous month-end
- `equity_change_pct`: change in equity inflow from last month, divided by the **size** of last month's inflow
  (equity inflows were negative in 2020; dividing by a negative number would flip the sign)
- `equity_change_cr` and `sip_change_cr`: the same changes in rupees (crore)
- `sip_share_pct`: SIP contribution as a share of equity net inflow. SIP money also goes into hybrid and index funds,
  so this is a rough guide, not an exact split

In [6]:
con.execute("""
CREATE OR REPLACE VIEW monthly AS
WITH joined AS (
    SELECT n.month,
           n.nifty_close,
           e.equity_inflow_cr,
           e.equity_aum_cr,
           s.sip_cr
    FROM nifty n
    LEFT JOIN equity e USING (month)
    LEFT JOIN sip s USING (month)
)
SELECT month,
       nifty_close,
       100 * (nifty_close / LAG(nifty_close) OVER w - 1)                                  AS nifty_return_pct,
       equity_inflow_cr,
       equity_inflow_cr - LAG(equity_inflow_cr) OVER w                                    AS equity_change_cr,
       100 * (equity_inflow_cr - LAG(equity_inflow_cr) OVER w) / ABS(LAG(equity_inflow_cr) OVER w) AS equity_change_pct,
       sip_cr,
       sip_cr - LAG(sip_cr) OVER w                                                        AS sip_change_cr,
       100 * (sip_cr / LAG(sip_cr) OVER w - 1)                                            AS sip_change_pct,
       100 * sip_cr / NULLIF(equity_inflow_cr, 0)                                         AS sip_share_pct,
       equity_aum_cr
FROM joined
WINDOW w AS (ORDER BY month)
ORDER BY month
""")
data = con.execute("SELECT * FROM monthly WHERE month >= '2019-04'").df()
print(len(data), "months")
data.tail()

91 months


,month,nifty_close,nifty_return_pct,equity_inflow_cr,equity_change_cr,equity_change_pct,sip_cr,sip_change_cr,sip_change_pct,sip_share_pct,equity_aum_cr
86,2026-06,23865.750000,1.350447,28973.413287,6065.640321,26.478525,31781.0,827.0,2.671706,109.690217,3.733731e+06
87,2026-07,24383.599609,2.169844,24697.385391,-4276.027896,-14.758454,31961.0,180.0,0.566376,129.410460,3.836192e+06
88,2026-08,24080.400391,-1.243456,29328.617931,4631.232540,18.751914,32297.0,336.0,1.051281,110.121111,3.921147e+06
89,2026-09,22620.449219,-6.062819,38306.552049,8977.934118,30.611514,32490.0,193.0,0.597579,84.815778,3.754275e+06
90,2026-10,23053.099609,1.912652,32993.637260,-5312.914789,-13.869468,32160.0,-330.0,-1.015697,97.473339,3.714550e+06


### Months when equity funds saw net outflows
These are the months where a percentage change is least meaningful, so we list them.

In [7]:
data = con.execute("SELECT month, equity_inflow_cr, nifty_return_pct FROM monthly WHERE equity_inflow_cr < 0 ORDER BY month").df()
data

,month,equity_inflow_cr,nifty_return_pct
0,2020-07,-2480.352095,7.487314
1,2020-08,-3999.619684,2.836061
2,2020-09,-734.403111,-1.228981
3,2020-10,-2724.954940,3.510548
4,2020-11,-12917.358875,11.394126
5,2020-12,-10147.115931,7.809420
6,2021-01,-9253.218250,-2.482882
7,2021-02,-4534.357289,6.560888


### Equity category mix by month

In [8]:
con.execute("""
CREATE OR REPLACE VIEW category_mix AS
SELECT month, category, net_inflow_cr,
       100 * net_inflow_cr / SUM(net_inflow_cr) OVER (PARTITION BY month) AS share_pct
FROM equity_categories
ORDER BY month, category
""")
data = con.execute("""
SELECT category, ROUND(SUM(net_inflow_cr)) AS total_inflow_cr
FROM category_mix
GROUP BY category
ORDER BY total_inflow_cr DESC
""").df()
data

,category,total_inflow_cr
0,Sectoral & thematic,300160.0
1,Flexi cap,240094.0
2,Small cap,221970.0
3,Mid cap,204053.0
4,Large & mid cap,177814.0
5,Multi cap,172905.0
6,"Value, contra, focused & dividend yield",126740.0
7,Large cap,79978.0
8,ELSS (tax saver),8593.0


In [9]:
con.execute("SELECT * FROM monthly").df().to_parquet(OUT / "monthly.parquet", index=False)
con.close()
print("saved data/monthly.parquet")

saved data/monthly.parquet
